In [ ]:
!pip install torch transformers datasets peft trl accelerate bitsandbytes swanlab

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 371.2/371.2 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.1/161.1 kB 19.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
!swanlab login

swanlab: You can find your API key at: https://swanlab.cn/space/~/settings
swanlab: Paste an API key from your profile and hit enter, or press 'CTRL + C' 
to quit: 
⠦ Waiting for the swanlab cloud response.
swanlab: Login successfully. Hi, lsy_8117!


In [ ]:
!pip install torch-tb-profiler

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.8 MB/s eta 0:00:00


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

import swanlab
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import gc
import os
import shutil
import json
import time
import math
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from torch.profiler import profile, record_function, ProfilerActivity, schedule
from collections import defaultdict


In [ ]:
model_id = "Qwen/Qwen2.5-3B"
dataset_name = "openai/gsm8k"
output_dir = "./qwen2.5-3b-distill-student"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def load_distillation_data(file_path, tokenizer):
    data_list = []
    max_observed_tokens = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            item = json.loads(line)

            # assemble the new distilled data
            reasoning = item['response']['reasoning']
            final_ans = item['response']['answer']
            full_text = (
                f"Question: {item['question']}\n"
                f"Answer: {reasoning}\n"
                f"#### {final_ans}{tokenizer.eos_token}"
            )

            # Count tokens
            tokens = tokenizer.encode(full_text, add_special_tokens=False)
            max_observed_tokens = max(max_observed_tokens, len(tokens))
            data_list.append({"text": full_text})

    final_max_length = 2 ** math.ceil(math.log2(max_observed_tokens))

    print(f"Max #(token) from dataset: {max_observed_tokens}") # Max(token)
    print(f"We'll set max_length as: {final_max_length}") # max_length

    distill_ds = Dataset.from_list(data_list)

    def tokenize_fn(examples):
        res = tokenizer(
            examples["text"],
            max_length=final_max_length,
            truncation=True,
            padding="max_length"
        )

        labels = []
        for ids in res["input_ids"]:
            label = [(l if l != tokenizer.pad_token_id else -100) for l in ids]
            labels.append(label)

        res["labels"] = labels
        return res

    return distill_ds.map(
        tokenize_fn,
        batched=True,
        remove_columns=["text"]
    )

In [ ]:
distill_dataset = load_distillation_data("/content/drive/MyDrive/HPML Project/SFT/success.jsonl", tokenizer)

Max #(token) from dataset: 357
We'll set max_length as: 512


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [ ]:
def train_and_profile(model, tokenized_train_3000, model_name, num_epochs=1, batch_size=4, num_workers=4, lr=3e-5):
    best_val_loss = float('inf')

    # train, val split
    tokenized_train_3000.set_format(type="torch", columns=["input_ids", "labels"])
    val_size = 300
    train_size = len(tokenized_train_3000) - val_size

    train_ds = Subset(tokenized_train_3000, range(train_size))
    val_ds = Subset(tokenized_train_3000, range(train_size, len(tokenized_train_3000)))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    # optimizer/scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

    print(f"Starting Training: {model_name} | Train: {train_size} | Val: {val_size}")

    for epoch in range(num_epochs):
        epoch_train_loss = 0.0
        epoch_data_time = 0.0
        epoch_gpu_time = 0.0
        epoch_throughput = 0.0
        last_lr = 0.0

        # train
        model.train()
        start_data_time = time.time()

        pbar = tqdm(train_loader, desc=f"Epoch {epoch} [Train]")
        for batch in pbar:

            data_loading_duration = time.time() - start_data_time

            batch = {k: v.to(model.device) for k, v in batch.items()}

            torch.cuda.synchronize()
            start_gpu_time = time.time()

            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()

            last_lr = scheduler.get_last_lr()[0]
            scheduler.step()

            torch.cuda.synchronize()
            gpu_duration = time.time() - start_gpu_time

            epoch_train_loss += loss.item()
            epoch_data_time += data_loading_duration
            epoch_gpu_time += gpu_duration
            epoch_throughput += (batch_size / (data_loading_duration + gpu_duration))

            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            start_data_time = time.time()

        cleanup()

        # eval
        model.eval()
        epoch_val_loss = 0.0
        with torch.no_grad():
            for val_batch in tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
                val_batch = {k: v.to(model.device) for k, v in val_batch.items()}
                val_outputs = model(**val_batch)
                epoch_val_loss += val_outputs.loss.item()

        cleanup()

        num_train_batches = len(train_loader)
        num_val_batches = len(val_loader)

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss

            model.save_pretrained(last_model_path)
            tokenizer.save_pretrained(last_model_path)

            print(f"Saved best model so far.")

        swanlab.log({
            "epoch": epoch,
            "train/avg_loss": epoch_train_loss / num_train_batches,
            "train/avg_lr": last_lr,
            "val/avg_loss": epoch_val_loss / num_val_batches,
            "profile/avg_data_loading_time": epoch_data_time / num_train_batches,
            "profile/avg_gpu_compute_time": epoch_gpu_time / num_train_batches,
            "profile/avg_throughput": num_train_batches / epoch_gpu_time,
        }, step=epoch)

        print(f"Epoch {epoch} Summary: Train Loss: {epoch_train_loss/num_train_batches:.4f} | Val Loss: {epoch_val_loss/num_val_batches:.4f}")

    print(f"Training for {model_name} completed.")

# Separate Profiling

## Model Set Up

In [ ]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    # task_type="CAUSAL_LM",
)

In [ ]:
model_distill = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    # attn_implementation="flash_attention_2",
)
model_distill = get_peft_model(model_distill, lora_config)
model_distill.print_trainable_parameters()
save_path = "./qwen2.5-3B-Distill-Orig"
last_model_path = save_path

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## SFT & Profiling Set Up

In [ ]:
BATCH_SIZE = 4
NUM_WORKERS = 4
LR = 3e-5
NUM_EPOCHS = 1
PIN_MEMORY = True
PROFILE_STEPS = 5

prof_schedule = schedule(wait=1, warmup=1, active=10, repeat=2)

In [ ]:
distill_dataset.set_format(type="torch", columns=["input_ids", "labels"])
train_ds = Subset(distill_dataset, range(len(distill_dataset) - 300))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

optimizer = torch.optim.AdamW(model_distill.parameters(), lr=LR, weight_decay=0.01)


In [ ]:
distill_dataset.set_format(type="torch", columns=["input_ids", "labels"])
train_ds = Subset(distill_dataset, range(len(distill_dataset) - 300))
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=4, pin_memory=True)
optimizer = torch.optim.AdamW(model_distill.parameters(), lr=3e-5, weight_decay=0.01)


STAGES = ["data_loading", "host_to_device", "forward", "backward", "optimizer"]

In [ ]:
step_records = []

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
    with_stack=True,
    with_flops=True,
) as prof:
    model_distill.train()
    loader_iter = iter(train_loader)

    for i in range(PROFILE_STEPS):
        stage_times = {}
        torch.cuda.reset_peak_memory_stats()

        # -- data loading --
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with record_function("data_loading"):
            batch = next(loader_iter)
        torch.cuda.synchronize()
        stage_times["data_loading"] = time.perf_counter() - t0

        # -- host to device --
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with record_function("host_to_device"):
            batch = {k: v.to(model.device, dtype=torch.bfloat16)
                     for k, v in batch.items() if v.is_floating_point()}
        torch.cuda.synchronize()
        stage_times["host_to_device"] = time.perf_counter() - t0

        # -- forward --
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with record_function("forward"):
            outputs = model_distill(**batch)
            loss = outputs.loss
        torch.cuda.synchronize()
        stage_times["forward"] = time.perf_counter() - t0

        # -- backward --
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with record_function("backward"):
            loss.backward()
        torch.cuda.synchronize()
        stage_times["backward"] = time.perf_counter() - t0

        # -- optimizer --
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with record_function("optimizer"):
            optimizer.step()
            optimizer.zero_grad()
        torch.cuda.synchronize()
        stage_times["optimizer"] = time.perf_counter() - t0

        peak_vram = torch.cuda.max_memory_allocated() / 1024**3
        reserved_vram = torch.cuda.max_memory_reserved() / 1024**3

        step_records.append({
            "step": i,
            "stage_times": stage_times,
            "peak_vram_gb": peak_vram,
            "reserved_vram_gb": reserved_vram,
        })

        prof.step()

In [ ]:
# ── PyTorch profiler kernel-level table ────────────────────────────────────
print("\n" + "="*80)
print("PYTORCH PROFILER — CUDA KERNEL BREAKDOWN (top 20 by cuda_time_total)")
print("="*80)
print(prof.key_averages().table(
    sort_by="cuda_time_total",
    row_limit=20
))

# ── per-stage averages across steps ───────────────────────────────────────
print("\n" + "="*80)
print("PER-STAGE TIME BREAKDOWN (avg over steps, in ms)")
print("="*80)

stage_totals = defaultdict(float)
for rec in step_records:
    for stage, t in rec["stage_times"].items():
        stage_totals[stage] += t

n = len(step_records)
total_avg = sum(stage_totals.values()) / n

print(f"{'Stage':<20} {'Avg (ms)':>10} {'% of step':>12}")
print("-" * 44)
for stage in STAGES:
    avg_ms = stage_totals[stage] / n * 1000
    pct = (stage_totals[stage] / n) / total_avg * 100
    print(f"{stage:<20} {avg_ms:>10.2f} {pct:>11.1f}%")
print("-" * 44)
print(f"{'TOTAL STEP':<20} {total_avg*1000:>10.2f} {'100.0%':>12}")

# ── GPU utilization proxy (gpu_time / wall_time) ───────────────────────────
print("\n" + "="*80)
print("GPU UTILIZATION PROXY (per step)")
print("="*80)
print(f"{'Step':<8} {'GPU time (ms)':>15} {'Wall time (ms)':>15} {'GPU util %':>12}")
print("-" * 52)

for rec in step_records:
    wall_ms = sum(rec["stage_times"].values()) * 1000
    # GPU-bound stages only (excludes cpu-only data_loading)
    gpu_ms = sum(rec["stage_times"][s] for s in ["forward", "backward", "optimizer"]) * 1000
    util = gpu_ms / wall_ms * 100
    print(f"{rec['step']:<8} {gpu_ms:>15.2f} {wall_ms:>15.2f} {util:>11.1f}%")

# ── VRAM ──────────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("PEAK VRAM (per step)")
print("="*80)
print(f"{'Step':<8} {'Peak alloc (GB)':>16} {'Peak reserved (GB)':>20}")
print("-" * 46)
for rec in step_records:
    print(f"{rec['step']:<8} {rec['peak_vram_gb']:>16.3f} {rec['reserved_vram_gb']:>20.3f}")

# ── memory traffic from profiler ──────────────────────────────────────────
print("\n" + "="*80)
print("MEMORY TRAFFIC — TOP OPS BY SELF CUDA MEMORY (profiler)")
print("="*80)
print(prof.key_averages().table(
    sort_by="self_cuda_memory_usage",
    row_limit=15
))

# ── export for TensorBoard ────────────────────────────────────────────────
prof.export_chrome_trace("/content/drive/MyDrive/HPML Project/SFT/trace.json")
print("\nTrace exported to /content/drive/MyDrive/HPML Project/SFT/trace.json")


PYTORCH PROFILER — CUDA KERNEL BREAKDOWN (top 20 by cuda_time_total)
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  Total KFLOPs  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                forward         0.00%       0.000us         0.00%       0.000us       0.000u

In [ ]:
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
    with_stack=True,
) as prof:
    model_distill.train()
    loader_iter = iter(train_loader)  # create iterator outside so next() is what we time

    for i in range(PROFILE_STEPS):
        with record_function("data_loading"):
            batch = next(loader_iter)

        with record_function("host_to_device"):
            batch = {k: v.to(model_distill.device) for k, v in batch.items()}

        with record_function("forward"):
            outputs = model_distill(**batch)
            loss = outputs.loss

        with record_function("backward"):
            loss.backward()

        with record_function("optimizer"):
            optimizer.step()
            optimizer.zero_grad()

        prof.step()

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=20))
prof.export_chrome_trace("./profiler_logs/trace.json")

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                forward         0.00%       0.000us         0.00%       0.000us       0.000us        2.339s       126.82%        2.339s     467.877ms           0 B           0 B           0 B           0 